<a href="https://colab.research.google.com/github/KenPham0604/US_Accident_Severity_Prediction/blob/main/Pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 — Setup
from google.colab import drive
import pandas as pd
import numpy as np
import zipfile
import shutil
import os
import gc

pd.set_option("display.max_columns", None)
SEED = 42
np.random.seed(SEED)

drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/ADY201m/'

print("✅ Setup completeD")

In [ ]:
# 2 — Defining shared functions: RAM optimization, chunked reading, and file saving.

def optimize_ram(df, verbose=True):
    ram_before = df.memory_usage(deep=True).sum() / (1024**2)
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif col_type == "object":
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype("category")
    ram_after = df.memory_usage(deep=True).sum() / (1024**2)
    if verbose:
        print(f"RAM: {ram_before:.2f} MB -> {ram_after:.2f} MB (describe {(1-ram_after/ram_before)*100:.1f}%)")
    return df


def read_zip_by_chunking(file_name, drive_folder=DRIVE_FOLDER, chunksize=500_000):
    drive_zip = f'{drive_folder}{file_name}.zip'
    local_zip = f'/content/{file_name}.zip'
    shutil.copy(drive_zip, local_zip)

    with zipfile.ZipFile(local_zip, 'r') as z:
        csv_name = z.namelist()[0]
        chunks_list = []
        with z.open(csv_name) as f:
            for i, chunk in enumerate(pd.read_csv(f, chunksize=chunksize)):
                chunks_list.append(chunk)
                print(f"  Chunk {i+1}: {sum(len(c) for c in chunks_list):,} rows")

    df = pd.concat(chunks_list, ignore_index=True)
    del chunks_list
    gc.collect()
    os.remove(local_zip)
    return df


def save_csv_as_zip(df_to_save, file_name, drive_folder=DRIVE_FOLDER):
    local_csv = f'/content/{file_name}.csv'
    local_zip = f'/content/{file_name}.zip'
    drive_zip = f'{drive_folder}{file_name}.zip'

    df_to_save.to_csv(local_csv, index=False)
    with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(local_csv, arcname=f'{file_name}.csv')
    shutil.copy(local_zip, drive_zip)

    shape_mb = os.path.getsize(local_zip) / (1024**2)
    print(f"✅ Saved: {drive_zip} ({df_to_save.shape[0]:,} rows, {shape_mb:.2f} MB zip)")

    os.remove(local_csv)
    os.remove(local_zip)
    gc.collect()
    return drive_zip

print("✅ Finished defining three shared functions: optimize_ram, read_zip_by_chunking, and save_csv_as_zip.")

In [ ]:
# 3 — Read the cleaned raw data
df = read_zip_by_chunking('US_Accidents_cleaned')
print(f"\nSize of input data: {df.shape}")

df = optimize_ram(df)
gc.collect()

print(f"\nData type:\n{df.dtypes}")

In [ ]:
# 4 — Run this cell if Cell 3 crashes, instead of re-running Cell 3.
df = read_zip_by_chunking('US_Accidents_cleaned', chunksize=200_000)
print(f"\nSize of input data: {df.shape}")
df = optimize_ram(df)
gc.collect()

In [ ]:
# 5 — Remove End_Time; transform datetime for Start_Time, Weather_Timestamp
if "End_Time" in df.columns:
    df = df.drop(columns=["End_Time"])
    print("Removed End_Time (leak data)")

df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["Weather_Timestamp"] = pd.to_datetime(df["Weather_Timestamp"], errors="coerce")

print(f"NaT ở Start_Time: {df['Start_Time'].isna().sum()}")
print(f"NaT ở Weather_Timestamp: {df['Weather_Timestamp'].isna().sum()}")

df["Start_Time"] = df["Start_Time"].ffill()
df["Weather_Timestamp"] = df["Weather_Timestamp"].ffill()

gc.collect()
print(f"\nShape after this step: {df.shape}")

In [ ]:
# 6 — Extract time variables from Start_Time

df["Year"] = df["Start_Time"].dt.year.astype("int16")
df["Month"] = df["Start_Time"].dt.month.astype("int8")
df["Hour"] = df["Start_Time"].dt.hour.astype("int8")
df["DayOfWeek"] = df["Start_Time"].dt.dayofweek.astype("int8")

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall",
}
df["Season"] = df["Month"].map(season_map).astype("category")

print(df[["Start_Time", "Year", "Month", "Hour", "DayOfWeek", "Season"]].head())
gc.collect()

In [ ]:
# 7 — Create additional sine and cosine columns for the cyclical variable.

df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24).astype("float32")
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24).astype("float32")

df["DayOfWeek_sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7).astype("float32")
df["DayOfWeek_cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7).astype("float32")

df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12).astype("float32")
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12).astype("float32")

print(df[["Hour", "Hour_sin", "Hour_cos", "DayOfWeek", "DayOfWeek_sin", "DayOfWeek_cos"]].head())
gc.collect()

In [ ]:
# 8 — Calculate Weather_Timestamp_diff_minutes, excluding the two raw datetime columns.
df["Weather_Timestamp_diff_minutes"] = (
    (df["Weather_Timestamp"] - df["Start_Time"]).dt.total_seconds() / 60
).astype("float32")

print(f"Time deviation statistics (minutes):")
print(df["Weather_Timestamp_diff_minutes"].describe())

df = df.drop(columns=["Start_Time", "Weather_Timestamp"])
gc.collect()

print(f"\nShape after this step: {df.shape}")

In [ ]:
# 9 — Remove columns that are not used at all (ID, Source, Description).

removed_cols = ["ID", "Source", "Description"]
df = df.drop(columns=[c for c in removed_cols if c in df.columns])

print(f"Shape after removing columns: {df.shape}")
print(f"List of current columns:\n{list(df.columns)}")
gc.collect()

In [ ]:
# 10 — Perform final RAM optimization and save the fully processed file.

df = optimize_ram(df)
gc.collect()

print(f"\nShape: {df.shape}")
print(f"List of columns:\n{list(df.columns)}")
print(f"\nData type:\n{df.dtypes}")

save_csv_as_zip(df, 'US_Accidents_processed_full')

print("\n✅ Completed — file 'US_Accidents_processed_full.zip' saved into Drive")